# BasketballGAN — Training on Google Colab

Train a VAE-GAN to generate defensive basketball play simulations.

> **Before starting:** Runtime → Change runtime type → **T4 GPU**

## 1. Clone & Install

In [ ]:
!git clone https://github.com/TQG1997/basketball.git
%cd basketball

# Colab only indexes TF >= 2.16 (Keras 3), which removes compat.v1.
# Install TF 2.15 from PyPI directly — last version with tf.layers / tf.Session.
!pip install -q tensorflow==2.15.1 --index-url https://pypi.org/simple/
!pip install -q numpy scipy matplotlib rdp protobuf packaging

## 2. Download Dataset

Downloads the required `.npy` files (~730MB) from Google Drive via `gdown`.

In [ ]:
import os

os.makedirs('data', exist_ok=True)

!pip install -q gdown

# Download entire dataset folder from Google Drive
# Folder: https://drive.google.com/drive/folders/1uNPw7LOA3xENclQRtSlUftiR7tlVNOts
FOLDER_ID = '1uNPw7LOA3xENclQRtSlUftiR7tlVNOts'
!gdown --folder https://drive.google.com/drive/folders/{FOLDER_ID} -O data/

# gdown saves to data/<foldername>/ — move files up one level
import glob, shutil
for f in glob.glob('data/*/*.npy'):
    shutil.move(f, 'data/')
# Clean up empty subdirectory
!rmdir data/*/ 2>/dev/null

print('\nDataset files:')
!ls -lh data/

## 3. Mount Google Drive

Save checkpoints and logs to your Google Drive so they persist after the Colab instance shuts down.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_OUTPUT = '/content/drive/MyDrive/basketballgan/tmp'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f'Output directory: {DRIVE_OUTPUT}')

## 4. Verify Setup

In [ ]:
import tensorflow as tf
import numpy as np

print('TensorFlow version:', tf.__version__)
print('NumPy version:', np.__version__)
print('GPU available:', tf.test.is_gpu_available())

# Verify dataset files exist
required = ['50Real.npy', '50Seq.npy', 'SeqCond.npy', 'RealCond.npy']
for f in required:
    path = f'data/{f}'
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f'  ✓ {f} ({size_mb:.1f} MB)')
    else:
        print(f'  ✗ {f} — MISSING! Download it first.')
        
# Quick data shape check
real = np.load('data/50Real.npy')
seq = np.load('data/50Seq.npy')
print(f'\n50Real.npy shape: {real.shape}')
print(f'50Seq.npy shape:  {seq.shape}')

## 5. Train

Key parameters you may want to adjust:
- `--max_epochs`: Stop after N epochs (Colab free tier runs ~12h max)
- `--batch_size`: Reduce if you hit OOM (128 → 64 or 32)
- `--beta`: KL divergence weight (higher = stronger VAE prior)
- `--recon_weight`: Reconstruction loss weight

In [ ]:
!python src/Train_Triple.py \
    --folder_path='{DRIVE_OUTPUT}' \
    --data_path='data' \
    --max_epochs=500 \
    --batch_size=64 \
    --beta=0.001 \
    --recon_weight=1.0 \
    --checkpoint_step=100 \
    --vis_freq=10

## 6. Monitor Training

Checkpoints saved to `{DRIVE_OUTPUT}/Checkpoints/`.
Sample animations saved to `{DRIVE_OUTPUT}/Samples/`.
TensorBoard logs at `{DRIVE_OUTPUT}/Log/`.

In [ ]:
# List saved checkpoints
!ls -lh {DRIVE_OUTPUT}/Checkpoints/ 2>/dev/null || echo 'No checkpoints yet'

# List sample animations
!ls -lh {DRIVE_OUTPUT}/Samples/ 2>/dev/null || echo 'No samples yet'

## 7. View Generated Samples

In [ ]:
from IPython.display import Video
import glob

# Play the latest generated animation
samples = sorted(glob.glob(f'{DRIVE_OUTPUT}/Samples/*.mp4'))
if samples:
    print(f'Latest sample: {samples[-1]}')
    Video(samples[-1], width=500)
else:
    print('No samples found. Training may still be in progress.')

---

## Resume Training

If your Colab session disconnects, re-run cells 1–4 above, then restore from the latest checkpoint:

In [ ]:
# Find latest checkpoint
import glob
checkpoints = sorted(glob.glob(f'{DRIVE_OUTPUT}/Checkpoints/model.ckpt-*.meta'))
if checkpoints:
    latest = checkpoints[-1].replace('.meta', '')
    print(f'Latest checkpoint: {latest}')
    # Copy checkpoint to local tmp and resume
    !mkdir -p tmp/Checkpoints
    !cp {DRIVE_OUTPUT}/Checkpoints/* tmp/Checkpoints/
    !python src/Train_Triple.py --folder_path='tmp' --data_path='data' --max_epochs=1000
else:
    print('No checkpoints found. Start training from scratch (Section 5).')